# Polarisation synthesis

- Open the San Francisco ALOS-1 scattering matrix.
- Convert `S` to `T3`.
- Apply 4 × 1 multilooking.
- Apply a 5 × 5 boxcar filter.
- Synthesize and display an RGB polarisation image.

In [ ]:
from pathlib import Path

from dask.diagnostics import ProgressBar

from polsarpro.io import open_netcdf_beam
from polsarpro.polarisation import polarisation_synthesis
from polsarpro.util import S_to_T3, boxcar, multilook

input_file = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")

In [ ]:
S = open_netcdf_beam(input_file)
S

In [ ]:
with ProgressBar():
    T3 = S_to_T3(S)
    T3 = multilook(T3, dim_az=4, dim_rg=1)
    T3 = boxcar(T3, dim_az=5, dim_rg=5)
    rgb = polarisation_synthesis(
        T3, phi=17.0, tau=-11.0, basis="sinclair"
    ).compute()

In [ ]:
high = rgb.quantile(0.98, dim=("y", "x")).drop_vars("quantile")
rgb_stretched = (rgb / high).clip(0, 1)
rgb_stretched.plot.imshow(rgb="band", figsize=(5, 7))